# Modelltraining

Ein Topic-Modell wird auf Basis von Redebeiträgen aus Plenarprotokollen des Deutschen Bundestages
im [CPP-BT][https://zenodo.org/records/18177196] trainiert und lokal gespeichert.



## Vorbereitung

Das Korpus herunterladen, sehr kurze und leere Texte (unter x Symbole) ausschließen und auf die
Jahre 2014-2025 begrenzen.

In [ ]:
import pandas as pd

from keyword_selection.data import load_corpus

df = load_corpus()


In [ ]:
MIN_LEN = 100
START_YEAR = "2014"
END_YEAR = "2025"

df = df[df.rede_text.str.len() >= MIN_LEN]

# set datetime index
timestamps = pd.to_datetime(df.sitzung_datum)
turns = df.set_index(timestamps).rede_text
turns = turns.sort_index()

turns = turns.loc[START_YEAR:END_YEAR]

turns.head()

## Training

Stoppwörter werden entfernt. Um Reproduzierbarkeit zu gewährleisten, wird UMAP manuell initialisiert
und ein `random_state` gesetzt sowie Parallelisierung mittels `n_jobs=1` deaktiviert. Die restlichen
Parameter von `UMAP` entsprechen den Werten, die von BERTopic [intern][1] gesetzt werden.

CountVectorizer muss ebenfalls manuell initialisiert werden, um die Standard-Settings anzupassen:

- Deutsche Stoppwörter übergeben
- N-Gram-Größe definieren (z.B. `ngram_range=(1, 2)` zum Einschließen von Bigrammen)
- Zu seltene Terme ausschließen (`min_df`)
- Zu häufige Terme ausschließen (`max_df`)

Weil wir möglichst viele potenziell interessante Zielwörter ermitteln möchten, setzen wir 
`top_n_words` bewusst viel höher als den Standardwert 10, um für jedes Topic eine breitere Auswahl
zu erhalten.

Das Training benötigt auf einem MacBook Pro mit M3-Prozessor und 24GB RAM nur ca. 5 Minuten.


[1]: https://github.com/MaartenGr/BERTopic/blob/9036123c97aaa9a6cc4bec4a6db1a7caf9209df6/bertopic/_bertopic.py#L268

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from stop_words import get_stop_words
from umap import UMAP

stop_words = get_stop_words("german")
random_state = 24601

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=random_state,
    n_jobs=1,
)

vectorizer_model = CountVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.8,
    lowercase=False,
)

topic_model = BERTopic(
    verbose=True,
    language="german",
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    top_n_words=50,
)
topics, probs = topic_model.fit_transform(turns.to_list())

Die vorsortierten Redebeiträge inkl. Datumsstempel und das trainierte Modell werden gespeichert.

In [ ]:
from keyword_selection.data import DATA_PATH

turns.to_frame().to_parquet(DATA_PATH / "output" / "preprocessed_corpus.parquet")
topic_model.save(
    DATA_PATH / "output" / "topic_model",
    serialization="safetensors",
    save_ctfidf=True,
)
